# 7-Eleven 식품류 상품 개수 분석

본 분석은 `B4_ITEM_DV_INFO.csv` 데이터를 바탕으로 비식품류(소모품, 기기, 잡화 등)를 제외하고 실제 먹을 수 있는 **식품류 상품**의 총 개수를 집계한 결과입니다.

### 1. 식품류 필터링 및 총 개수 확인
비식품 대분류('소모품')와 상품명 내 비식품 키워드(텀블러, 머그, 봉투 등)를 제외하는 로직입니다.

In [1]:
import pandas as pd

# 데이터 로드
file_path = '../Database/B4_ITEM_DV_INFO.csv'
df = pd.read_csv(file_path)

# 비식품류 필터링 기준
non_food_keywords = ['텀블러', '머그', '커피밀', '여과지', '그라인더', '티스픈', '잔세트', '유리머그', '포장Tray', '봉투', '박스', '금액키', '쿠폰', '소모품', 'Tray']

is_food = (df['ITEM_LRDV_NM'] != '소모품') & (~df['ITEM_NM'].str.contains('|'.join(non_food_keywords), na=False))
df_food = df[is_food].copy()

print("=== 분석 결과 ===")
print(f"1. 전체 원천 데이터 상품 수: {len(df):,} 개")
print(f"2. 식품류 상품 수 (최종): {len(df_food):,} 개")
print(f"3. 제외된 비식품류 상품 수: {len(df) - len(df_food):,} 개")

=== 분석 결과 ===
1. 전체 원천 데이터 상품 수: 92,145 개
2. 식품류 상품 수 (최종): 89,027 개
3. 제외된 비식품류 상품 수: 3,118 개


### 2. 식품류 대분류별 상세 현황
필터링된 식품류 중 상위 10개 카테고리의 구성입니다.

In [2]:
top_categories = df_food['ITEM_LRDV_NM'].value_counts().head(10)
for cat, count in top_categories.items():
    print(f"{cat}: {count:,}개")

과자: 21,271개
신선: 6,758개
미반: 6,396개
빵: 5,191개
음료: 4,677개
건강/기호식품: 4,309개
냉장: 4,223개
양주와인: 3,981개
조미료/건물: 3,928개
유음료: 3,810개


### 3. 모델별 API 비용

In [1]:
import pandas as pd

# 모델별 100만 토큰당 단가 데이터 (달러 기준)
data = {
    'Model': [
        'Llama 3.2 11B Vision', 
        'Gemini 1.5 Flash', 
        'DeepSeek-V3', 
        'Qwen-VL-Plus', 
        'GPT-4o-mini', 
        'Claude 3 Haiku', 
        'Qwen-VL-Max'
    ],
    'Input Price(1M)': [0.18, 0.075, 0.14, 0.15, 0.15, 0.25, 0.40],
    'Output Price(1M)': [0.18, 0.30, 0.28, 0.45, 0.60, 1.25, 1.20]
}

# 데이터프레임 생성
df = pd.DataFrame(data)

# 예상 토큰 수 설정 (백만 단위)
input_tokens = 31.15
output_tokens = 44.51

# 비용 계산 수식 적용
df['Input Cost($)'] = df['Input Price(1M)'] * input_tokens
df['Output Cost($)'] = df['Output Price(1M)'] * output_tokens
df['Total Cost($)'] = df['Input Cost($)'] + df['Output Cost($)']

# 총 비용이 저렴한 순서대로 정렬
df = df.sort_values(by='Total Cost($)')

# 보기 좋게 소수점 둘째 자리까지 포맷팅
pd.options.display.float_format = '{:.2f}'.format

# 결과 출력
print("가성비 AI 비전 모델 API 비용 비교 (입력 31.15M / 출력 44.51M 기준)\n")
print(df.to_string(index=False))

가성비 AI 비전 모델 API 비용 비교 (입력 31.15M / 출력 44.51M 기준)

               Model  Input Price(1M)  Output Price(1M)  Input Cost($)  Output Cost($)  Total Cost($)
Llama 3.2 11B Vision             0.18              0.18           5.61            8.01          13.62
    Gemini 1.5 Flash             0.07              0.30           2.34           13.35          15.69
         DeepSeek-V3             0.14              0.28           4.36           12.46          16.82
        Qwen-VL-Plus             0.15              0.45           4.67           20.03          24.70
         GPT-4o-mini             0.15              0.60           4.67           26.71          31.38
      Claude 3 Haiku             0.25              1.25           7.79           55.64          63.42
         Qwen-VL-Max             0.40              1.20          12.46           53.41          65.87
